# Watcher - Performance Analysis in ComCam On-Sky Campaign

This analysis is part of the preparation for the **LSST Camara On-Sky Workshop**. The goal is to evaluate the performance of the **Watcher** during the **ComCam On-Sky campaign**, which took place from **October 24, 2024, to December 11, 2024**. 

This notebook focuses on a specific subtask: **Watcher - Response time**. What was the average response time to an alarm? This means that the alarm was either acknowledged or that the issue was resolved after it was triggered (or any level or “no alarm”). This can be a histogram or some sort of statistical metric. 

Laura Toribio 13-05-2025

In [ ]:
from astropy.time import Time

#from lsst.sitcom.vandv.logger import create_logger
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import pandas as pd

## Information about the Watcher

In [ ]:
# Create an EFD client instance
client = makeEfdClient()

In [ ]:
# make a list of all topics in the EFD related to Watcher
topics = await client.get_topics()
for topic in topics:
    if 'Watcher' in topic:
        print(topic)

In [ ]:
# get all fields related to the Watcher mute
await client.get_fields('lsst.sal.Watcher.logevent_alarm')

In [ ]:
# Get the duration data from October 24, 2024, to December 11, 2024.
start = Time("2024-10-24T00:00:00Z", scale="utc")
end = Time("2024-12-11T00:00:00Z", scale="utc")

watchers = await client.select_time_series(
                       "lsst.sal.Watcher.logevent_alarm", 
                      "*", 
                      start, 
                      end
)

# Watcher Response Time

In [ ]:
# Number of Watcher
num_watcher = len(watchers)
print(f"Number of watcher: {num_watcher}")

In [ ]:
# Convert Unix TAI timestamps to ISO 8601 format for better readability and datetime operations
watchers['private_efdStamp'] = pd.to_datetime(watchers['private_efdStamp'], unit="s")

watchers['timestampAcknowledged'] = pd.to_datetime(watchers['timestampAcknowledged'], unit="s")
watchers['timestampAutoAcknowledge'] = pd.to_datetime(watchers['timestampAutoAcknowledge'], unit="s")

In [ ]:
# Calculate the watcher respnse Time for each row
# Aux Columns
watchers['response_time'] = pd.NaT
watchers['severity_type'] = None

# Epoch time
epoch_time = pd.to_datetime('1970-01-01 00:00:00')

for i in range(len(watchers)):
    row = watchers.iloc[i]
    current_time = row['private_efdStamp']
    acknowledged = row['acknowledged']
    ts_ack = row['timestampAcknowledged']
    ts_auto = row['timestampAutoAcknowledge']
    severity = row['severity']
    maxSeverity = row['maxSeverity']
    reason = row['reason']

    # CASE 1: maxSeverity == 1 (o alarmas tipo "None")
    if maxSeverity == 1 or pd.isna(maxSeverity):
        if acknowledged:
            watchers.at[watchers.index[i], 'response_time'] = ts_ack - current_time
        else:
            if ts_auto != epoch_time:
                watchers.at[watchers.index[i], 'response_time'] = ts_auto - current_time
            else: 
                watchers.at[watchers.index[i], 'response_time'] = pd.NaT
        watchers.at[watchers.index[i], 'severity_type'] = 'None'
    
    # CASE 2: Alarm with response
    elif not acknowledged and ts_auto == epoch_time:
        watchers.at[watchers.index[i], 'response_time'] = pd.NaT
        watchers.at[watchers.index[i], 'severity_type'] = 'Alarm with response'
    
    # CASE 3: acknowledged = True y maxSeverity != 1
    elif acknowledged and maxSeverity != 1:
        start_idx = max(0, i - 4)
        alarm_before = watchers.iloc[start_idx:i]
        mask = (
            (alarm_before['reason'] == reason) &
            (alarm_before['acknowledged'] == False) &
            (alarm_before['severity'] >= severity)
        )
        alarm_filtered = alarm_before[mask]
        if not alarm_filtered.empty:
            fila_objetivo = alarm_filtered.iloc[0]
            time = ts_ack - fila_objetivo['private_efdStamp']
            watchers.at[watchers.index[i], 'response_time'] = time
            watchers.at[watchers.index[i], 'severity_type'] = 'Alarm acknowledged'

    # CASE 4: acknowledged = False y ts_auto válido
    elif not acknowledged and pd.notna(ts_auto) and ts_auto != epoch_time:
        start_idx = max(0, i - 10)
        alarm_before = watchers.iloc[start_idx:i]
        mask = (
            (alarm_before['reason'] == reason) &
            (alarm_before['acknowledged'] == False) &
            (alarm_before['severity'] == maxSeverity)
        )
        alarm_filtered = alarm_before[mask]
        if not alarm_filtered.empty:
            fila_objetivo = alarm_filtered.iloc[0]
            time = ts_auto - fila_objetivo['private_efdStamp']
            watchers.at[watchers.index[i], 'response_time'] = time
            watchers.at[watchers.index[i], 'severity_type'] = 'Alarm auto acknowledged'


NONE = 1
WARNING = 2
SERIOUS = 3
CRITICAL = 4

In [ ]:
# Ensure response_time is a timedelta
watchers['response_time'] = pd.to_timedelta(watchers['response_time'], errors='coerce')

# Convert timedelta to seconds
watchers['response_time_sec'] = watchers['response_time'].dt.total_seconds()

# Filtrar solo alarmas con respuesta válida
filtered = watchers[
    watchers['severity_type'].isin(['Alarm acknowledged', 'Alarm auto acknowledged']) &
    watchers['response_time_sec'].notna()
]

# General Stats
average_time = filtered['response_time_sec'].mean()
median_time = filtered['response_time_sec'].median()
std_dev_time = filtered['response_time_sec'].std()

print("General Stats:")
print("Average:", round(average_time, 2), "seconds")
print("Median:", round(median_time, 2), "seconds")
print("Std Dev:", round(std_dev_time, 2), "seconds")

# Stats for Alarma acknowledged and auto acknowledged
for alarm_type in ['Alarm acknowledged', 'Alarm auto acknowledged']:
    group = filtered[filtered['severity_type'] == alarm_type]
    avg = group['response_time_sec'].mean()
    median = group['response_time_sec'].median()
    std = group['response_time_sec'].std()
    
    print(f"\n Stats for {alarm_type}:")
    print("Average:", round(avg, 2), "seconds")
    print("Median:", round(median, 2), "seconds")
    print("Std Dev:", round(std, 2), "seconds")

# Histrogram
plt.figure(figsize=(10, 5))
sns.histplot(filtered['response_time_sec'], bins=50, kde=False, color='skyblue')
plt.title('Overall Response Time (Acknowledged + Auto Acknowledged)')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.axvline(average_time, color='red', linestyle='--', label=f'Avg = {round(average_time, 2)}s')
plt.axvline(median_time, color='green', linestyle='--', label=f'Median = {round(median_time, 2)}s')
plt.legend()
plt.tight_layout()
plt.show()

# --- Histogram: Alarm acknowledged ---
acknowledged_only = filtered[filtered['severity_type'] == 'Alarm acknowledged']
plt.figure(figsize=(10, 5))
sns.histplot(acknowledged_only['response_time_sec'], bins=40, kde=False, color='blue')
plt.title('Response Time – Alarm Acknowledged')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.tight_layout()
plt.show()

# --- Histogram: Alarm auto acknowledged ---
auto_only = filtered[filtered['severity_type'] == 'Alarm auto acknowledged']
plt.figure(figsize=(10, 5))
sns.histplot(auto_only['response_time_sec'], bins=40, kde=False, color='green')
plt.title('Response Time – Alarm Auto Acknowledged')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Number of Alarms')
plt.tight_layout()
plt.show()
